# 00 - URDF Setup & Forward Kinematics Validation

**RAPP Lab 04 - Vision-Action Model**

This notebook performs the one-time setup required before processing rosbags:

1. **Extract URDF** from `/robot_description` topic and save to `config/tm12s.urdf`
2. **Extract static transforms** from `/tf_static` topic
3. **Validate forward kinematics** implementation using the extracted URDF
4. **Visualize** the robot at zero configuration and sample joint angles

---

## 1. Imports and Configuration

In [18]:
import os
import sys
from pathlib import Path

# Ensure vam_utils is importable
workspace_dir = Path('/workspace')
if str(workspace_dir) not in sys.path:
    sys.path.insert(0, str(workspace_dir))

import numpy as np
import sqlite3
from typing import Optional, Tuple, List, Dict, Any

# ROS2 message deserialization
from rclpy.serialization import deserialize_message
from std_msgs.msg import String
from tf2_msgs.msg import TFMessage
from sensor_msgs.msg import JointState

# URDF parsing and kinematics
from urdf_parser_py.urdf import URDF

# Visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Imports successful!")

Imports successful!


In [19]:
# Configuration paths
ROSBAG_DIR = Path('/data/rosbags')
CONFIG_DIR = Path('/config')
ROSBAG_NAME = '26_04_03_RAPP_M_R2G1_01'

# Construct full path to rosbag database
rosbag_path = ROSBAG_DIR / ROSBAG_NAME
db_file = list(rosbag_path.glob('*.db3'))[0] if rosbag_path.exists() else None

print(f"Rosbag directory: {rosbag_path}")
print(f"Database file: {db_file}")
print(f"Config directory: {CONFIG_DIR}")

# Verify paths
assert rosbag_path.exists(), f"Rosbag directory not found: {rosbag_path}"
assert db_file is not None and db_file.exists(), f"No .db3 file found in {rosbag_path}"

Rosbag directory: /data/rosbags/26_04_03_RAPP_M_R2G1_01
Database file: /data/rosbags/26_04_03_RAPP_M_R2G1_01/26_04_03_RAPP_M_R2G1_01_0.db3
Config directory: /config


## 2. Helper Functions for Rosbag Reading

In [20]:
def get_topic_messages(db_path: Path, topic_name: str, limit: Optional[int] = None) -> List[bytes]:
    """
    Extract raw serialized messages from a rosbag database for a given topic.
    
    Args:
        db_path: Path to the .db3 file
        topic_name: The ROS topic to extract
        limit: Maximum number of messages to retrieve (None for all)
        
    Returns:
        List of serialized message bytes
    """
    conn = sqlite3.connect(str(db_path))
    cursor = conn.cursor()
    
    # Get topic ID
    cursor.execute("SELECT id FROM topics WHERE name = ?", (topic_name,))
    result = cursor.fetchone()
    if result is None:
        conn.close()
        raise ValueError(f"Topic '{topic_name}' not found in rosbag")
    
    topic_id = result[0]
    
    # Get messages
    query = "SELECT data FROM messages WHERE topic_id = ? ORDER BY timestamp"
    if limit:
        query += f" LIMIT {limit}"
    
    cursor.execute(query, (topic_id,))
    messages = [row[0] for row in cursor.fetchall()]
    
    conn.close()
    return messages


def list_topics(db_path: Path) -> Dict[str, Dict[str, Any]]:
    """
    List all topics in a rosbag with their message counts and types.
    
    Args:
        db_path: Path to the .db3 file
        
    Returns:
        Dictionary mapping topic names to info dicts
    """
    conn = sqlite3.connect(str(db_path))
    cursor = conn.cursor()
    
    cursor.execute("""
        SELECT t.name, t.type, COUNT(m.id) as msg_count
        FROM topics t
        LEFT JOIN messages m ON t.id = m.topic_id
        GROUP BY t.id
    """)
    
    topics = {}
    for name, msg_type, count in cursor.fetchall():
        topics[name] = {'type': msg_type, 'count': count}
    
    conn.close()
    return topics


# List available topics
topics = list_topics(db_file)
print("Available topics:")
for name, info in sorted(topics.items()):
    print(f"  {name}: {info['type']} ({info['count']} messages)")

Available topics:
  /joint_states: sensor_msgs/msg/JointState (32141 messages)
  /robot_description: std_msgs/msg/String (1 messages)
  /tf: tf2_msgs/msg/TFMessage (36823 messages)
  /tf_static: tf2_msgs/msg/TFMessage (3 messages)
  /zed/zed_description: std_msgs/msg/String (1 messages)
  /zed/zed_node/body_trk/skeletons: zed_msgs/msg/ObjectsStamped (7197 messages)
  /zed/zed_node/rgb/color/rect/camera_info: sensor_msgs/msg/CameraInfo (7197 messages)
  /zed/zed_node/rgb/color/rect/image: sensor_msgs/msg/Image (7197 messages)


## 3. Extract URDF from `/robot_description`

In [21]:
# Extract the robot_description message
robot_desc_messages = get_topic_messages(db_file, '/robot_description', limit=1)
print(f"Found {len(robot_desc_messages)} robot_description message(s)")

# Deserialize the message
robot_desc_msg = deserialize_message(robot_desc_messages[0], String)
urdf_string = robot_desc_msg.data

print(f"\nURDF string length: {len(urdf_string)} characters")
print(f"\nFirst 500 characters of URDF:\n{urdf_string[:500]}...")

Found 1 robot_description message(s)

URDF string length: 11712 characters

First 500 characters of URDF:
<?xml version="1.0" ?><!-- =================================================================================== --><!-- |    This document was autogenerated by xacro from /tm2_ws/install/tm12s_moveit_config/share/tm12s_moveit_config/config/tm12s.urdf.xacro | --><!-- |    EDITING THIS FILE BY HAND IS NOT RECOMMENDED                                 | --><!-- =================================================================================== --><robot name="tm12s"><gazebo reference="link_1"><selfCol...


In [22]:
# Save URDF to config directory
# Note: /config is mounted read-only in Docker, so we save to a writable location first
# In production, you'd copy this file to the config directory on the host

urdf_output_path = Path('/data/processed/tm12s.urdf')
urdf_output_path.parent.mkdir(parents=True, exist_ok=True)

with open(urdf_output_path, 'w') as f:
    f.write(urdf_string)

print(f"URDF saved to: {urdf_output_path}")
print(f"\nTo copy to config directory (run on host):")
print(f"  cp /home/maleen/csvdata/rapplab04/tm12s.urdf /home/maleen/git/RAPP_LAB_04/config/")

URDF saved to: /data/processed/tm12s.urdf

To copy to config directory (run on host):
  cp /home/maleen/csvdata/rapplab04/tm12s.urdf /home/maleen/git/RAPP_LAB_04/config/


In [23]:
# Parse the URDF
robot = URDF.from_xml_string(urdf_string)

print(f"Robot name: {robot.name}")
print(f"\nLinks ({len(robot.links)}):")
for link in robot.links:
    print(f"  - {link.name}")

print(f"\nJoints ({len(robot.joints)}):")
for joint in robot.joints:
    if joint.type != 'fixed':
        limits = f"[{joint.limit.lower:.3f}, {joint.limit.upper:.3f}]" if joint.limit else "no limits"
        print(f"  - {joint.name} ({joint.type}): {joint.parent} -> {joint.child}, limits: {limits}")

Robot name: tm12s

Links (9):
  - link_0
  - link_1
  - link_2
  - link_3
  - link_4
  - link_5
  - link_6
  - base
  - flange

Joints (8):
  - joint_1 (revolute): link_0 -> link_1, limits: [-6.283, 6.283]
  - joint_2 (revolute): link_1 -> link_2, limits: [-6.283, 6.283]
  - joint_3 (revolute): link_2 -> link_3, limits: [-2.827, 2.827]
  - joint_4 (revolute): link_3 -> link_4, limits: [-6.283, 6.283]
  - joint_5 (revolute): link_4 -> link_5, limits: [-6.283, 6.283]
  - joint_6 (revolute): link_5 -> link_6, limits: [-6.283, 6.283]


Unknown tag "ros2_control" in /robot[@name='tm12s']


## 4. Extract Static Transforms from `/tf_static`

In [24]:
# Extract tf_static messages
tf_static_messages = get_topic_messages(db_file, '/tf_static')
print(f"Found {len(tf_static_messages)} tf_static message(s)")

# Deserialize and collect all transforms
static_transforms = []
for msg_bytes in tf_static_messages:
    tf_msg = deserialize_message(msg_bytes, TFMessage)
    for transform in tf_msg.transforms:
        static_transforms.append({
            'parent': transform.header.frame_id,
            'child': transform.child_frame_id,
            'translation': [
                transform.transform.translation.x,
                transform.transform.translation.y,
                transform.transform.translation.z
            ],
            'rotation': [
                transform.transform.rotation.x,
                transform.transform.rotation.y,
                transform.transform.rotation.z,
                transform.transform.rotation.w
            ]
        })

print("\nStatic transforms:")
for tf in static_transforms:
    print(f"\n  {tf['parent']} -> {tf['child']}")
    print(f"    Translation: [{tf['translation'][0]:.4f}, {tf['translation'][1]:.4f}, {tf['translation'][2]:.4f}]")
    print(f"    Rotation (quat xyzw): [{tf['rotation'][0]:.4f}, {tf['rotation'][1]:.4f}, {tf['rotation'][2]:.4f}, {tf['rotation'][3]:.4f}]")

Found 3 tf_static message(s)

Static transforms:

  zed_camera_link -> zed_camera_center
    Translation: [0.0000, 0.0000, 0.0150]
    Rotation (quat xyzw): [0.0000, 0.0000, 0.0000, 1.0000]

  zed_camera_center -> zed_left_camera_frame
    Translation: [-0.0100, 0.0600, 0.0000]
    Rotation (quat xyzw): [0.0000, 0.0000, 0.0000, 1.0000]

  zed_left_camera_frame -> zed_left_camera_frame_optical
    Translation: [0.0000, 0.0000, 0.0000]
    Rotation (quat xyzw): [0.5000, -0.5000, 0.5000, -0.5000]

  zed_camera_center -> zed_right_camera_frame
    Translation: [-0.0100, -0.0600, 0.0000]
    Rotation (quat xyzw): [0.0000, 0.0000, 0.0000, 1.0000]

  zed_right_camera_frame -> zed_right_camera_frame_optical
    Translation: [0.0000, 0.0000, 0.0000]
    Rotation (quat xyzw): [0.5000, -0.5000, 0.5000, -0.5000]

  map -> base
    Translation: [3.6000, -0.2700, -0.2500]
    Rotation (quat xyzw): [0.0000, 0.0000, 1.0000, 0.0008]

  base -> link_0
    Translation: [0.0000, 0.0000, 0.0000]
    Rotati

In [25]:
# Save static transforms to a YAML file for reference
import yaml

transforms_output_path = Path('/data/processed/static_transforms.yaml')

transforms_data = {
    'source_rosbag': ROSBAG_NAME,
    'static_transforms': static_transforms
}

with open(transforms_output_path, 'w') as f:
    yaml.dump(transforms_data, f, default_flow_style=False)

print(f"Static transforms saved to: {transforms_output_path}")

Static transforms saved to: /data/processed/static_transforms.yaml


## 5. Forward Kinematics Implementation

We implement forward kinematics using the extracted URDF to compute the position of each link given joint angles.

In [26]:
def quaternion_to_rotation_matrix(q: np.ndarray) -> np.ndarray:
    """
    Convert quaternion (xyzw) to 3x3 rotation matrix.
    
    Args:
        q: Quaternion as [x, y, z, w]
        
    Returns:
        3x3 rotation matrix
    """
    x, y, z, w = q
    
    # Normalize quaternion
    norm = np.sqrt(x*x + y*y + z*z + w*w)
    x, y, z, w = x/norm, y/norm, z/norm, w/norm
    
    return np.array([
        [1 - 2*(y*y + z*z), 2*(x*y - z*w), 2*(x*z + y*w)],
        [2*(x*y + z*w), 1 - 2*(x*x + z*z), 2*(y*z - x*w)],
        [2*(x*z - y*w), 2*(y*z + x*w), 1 - 2*(x*x + y*y)]
    ])


def rpy_to_rotation_matrix(rpy: Tuple[float, float, float]) -> np.ndarray:
    """
    Convert roll-pitch-yaw angles to 3x3 rotation matrix.
    Uses XYZ (fixed axis) convention: R = Rz(yaw) * Ry(pitch) * Rx(roll)
    
    Args:
        rpy: Tuple of (roll, pitch, yaw) in radians
        
    Returns:
        3x3 rotation matrix
    """
    roll, pitch, yaw = rpy
    
    cr, sr = np.cos(roll), np.sin(roll)
    cp, sp = np.cos(pitch), np.sin(pitch)
    cy, sy = np.cos(yaw), np.sin(yaw)
    
    return np.array([
        [cy*cp, cy*sp*sr - sy*cr, cy*sp*cr + sy*sr],
        [sy*cp, sy*sp*sr + cy*cr, sy*sp*cr - cy*sr],
        [-sp, cp*sr, cp*cr]
    ])


def rotation_matrix_from_axis_angle(axis: np.ndarray, angle: float) -> np.ndarray:
    """
    Create rotation matrix from axis-angle representation (Rodrigues formula).
    
    Args:
        axis: 3D unit vector representing rotation axis
        angle: Rotation angle in radians
        
    Returns:
        3x3 rotation matrix
    """
    axis = np.array(axis)
    axis = axis / np.linalg.norm(axis)  # Normalize
    
    K = np.array([
        [0, -axis[2], axis[1]],
        [axis[2], 0, -axis[0]],
        [-axis[1], axis[0], 0]
    ])
    
    return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)


class ForwardKinematics:
    """
    Forward kinematics for robot arm using extracted URDF.
    """
    
    def __init__(self, urdf: URDF):
        self.urdf = urdf
        self.joint_names = []
        self.joint_info = {}
        self.link_names = [link.name for link in urdf.links]
        
        # Build kinematic chain for revolute joints
        for joint in urdf.joints:
            if joint.type == 'revolute' or joint.type == 'continuous':
                self.joint_names.append(joint.name)
                self.joint_info[joint.name] = {
                    'parent': joint.parent,
                    'child': joint.child,
                    'origin_xyz': joint.origin.xyz if joint.origin else [0, 0, 0],
                    'origin_rpy': joint.origin.rpy if joint.origin else [0, 0, 0],
                    'axis': joint.axis if joint.axis else [0, 0, 1],
                    'limits': (joint.limit.lower, joint.limit.upper) if joint.limit else (-np.pi, np.pi)
                }
        
        print(f"Initialized FK with {len(self.joint_names)} joints:")
        for i, name in enumerate(self.joint_names):
            info = self.joint_info[name]
            print(f"  {i}: {name} ({info['parent']} -> {info['child']})")
    
    def compute_transforms(self, joint_angles: Dict[str, float]) -> Dict[str, np.ndarray]:
        """
        Compute 4x4 homogeneous transforms for all links.
        
        Args:
            joint_angles: Dictionary mapping joint names to angles (radians)
            
        Returns:
            Dictionary mapping link names to 4x4 transforms from base
        """
        transforms = {}
        
        # Start with identity at base
        base_link = self.urdf.get_root()
        transforms[base_link] = np.eye(4)
        
        # Process joints — multi-pass to handle any ordering in URDF
        remaining = list(self.urdf.joints)
        for _pass in range(len(remaining) + 1):
            next_remaining = []
            for joint in remaining:
                parent = joint.parent
                child = joint.child

                if parent not in transforms:
                    next_remaining.append(joint)
                    continue

                # Build transform from parent to child
                T = np.eye(4)
                if joint.origin:
                    T[:3, 3] = joint.origin.xyz
                    T[:3, :3] = rpy_to_rotation_matrix(joint.origin.rpy)

                if joint.type in ['revolute', 'continuous']:
                    angle = joint_angles.get(joint.name, 0.0)
                    axis = joint.axis if joint.axis else [0, 0, 1]
                    R_joint = rotation_matrix_from_axis_angle(axis, angle)
                    T[:3, :3] = T[:3, :3] @ R_joint

                transforms[child] = transforms[parent] @ T

            remaining = next_remaining
            if not remaining:
                break

        return transforms
    
    def get_link_positions(self, joint_angles: Dict[str, float]) -> Dict[str, np.ndarray]:
        """
        Get 3D positions of all links.
        
        Args:
            joint_angles: Dictionary mapping joint names to angles (radians)
            
        Returns:
            Dictionary mapping link names to [x, y, z] positions
        """
        transforms = self.compute_transforms(joint_angles)
        return {name: T[:3, 3] for name, T in transforms.items()}
    
    def get_joint_positions(self, joint_angles: Dict[str, float]) -> List[np.ndarray]:
        """
        Get positions of joint origins (for visualization of kinematic chain).
        
        Args:
            joint_angles: Dictionary mapping joint names to angles (radians)
            
        Returns:
            List of [x, y, z] positions for each joint in order
        """
        transforms = self.compute_transforms(joint_angles)
        positions = []
        
        # Get base position
        base = self.urdf.get_root()
        positions.append(transforms[base][:3, 3])
        
        # Get position after each joint
        for joint_name in self.joint_names:
            child = self.joint_info[joint_name]['child']
            if child in transforms:
                positions.append(transforms[child][:3, 3])
        
        return positions


# Initialize forward kinematics
fk = ForwardKinematics(robot)

Initialized FK with 6 joints:
  0: joint_1 (link_0 -> link_1)
  1: joint_2 (link_1 -> link_2)
  2: joint_3 (link_2 -> link_3)
  3: joint_4 (link_3 -> link_4)
  4: joint_5 (link_4 -> link_5)
  5: joint_6 (link_5 -> link_6)


## 6. Validate Forward Kinematics

In [27]:
# Test with zero configuration
zero_joints = {name: 0.0 for name in fk.joint_names}
positions_zero = fk.get_joint_positions(zero_joints)

print("Joint positions at zero configuration:")
for i, pos in enumerate(positions_zero):
    label = "base" if i == 0 else fk.joint_names[i-1]
    print(f"  {label}: [{pos[0]:.4f}, {pos[1]:.4f}, {pos[2]:.4f}]")

Joint positions at zero configuration:
  base: [0.0000, 0.0000, 0.0000]
  joint_1: [0.0000, 0.0000, 0.1652]
  joint_2: [0.0000, 0.0000, 0.1652]
  joint_3: [0.0000, 0.0000, 0.8013]
  joint_4: [0.0000, -0.1818, 1.3337]
  joint_5: [0.0000, -0.1818, 1.4652]
  joint_6: [0.0000, -0.3167, 1.4652]


In [28]:
# Get a sample joint state from the rosbag for validation
joint_state_messages = get_topic_messages(db_file, '/joint_states', limit=10)
print(f"Found {len(joint_state_messages)} sample joint_state messages")

# Deserialize first message
sample_js = deserialize_message(joint_state_messages[0], JointState)

print(f"\nSample joint state:")
print(f"  Joint names: {list(sample_js.name)}")
print(f"  Positions (rad): {[f'{p:.4f}' for p in sample_js.position]}")
print(f"  Positions (deg): {[f'{np.degrees(p):.1f}' for p in sample_js.position]}")

Found 10 sample joint_state messages

Sample joint state:
  Joint names: ['joint_1', 'joint_2', 'joint_3', 'joint_4', 'joint_5', 'joint_6']
  Positions (rad): ['-1.5709', '0.9640', '1.9665', '0.6598', '1.5504', '-0.0788']
  Positions (deg): ['-90.0', '55.2', '112.7', '37.8', '88.8', '-4.5']


In [29]:
# Map joint state to FK joint names
# Create mapping between joint_states names and URDF joint names
sample_joint_angles = {}
for i, name in enumerate(sample_js.name):
    # Find matching joint in URDF (handle naming differences)
    for urdf_joint in fk.joint_names:
        if name in urdf_joint or urdf_joint in name or name.replace('_joint', '') in urdf_joint:
            sample_joint_angles[urdf_joint] = sample_js.position[i]
            break
    else:
        # Direct match attempt
        if name in fk.joint_names:
            sample_joint_angles[name] = sample_js.position[i]

print("Mapped joint angles:")
for name, angle in sample_joint_angles.items():
    print(f"  {name}: {angle:.4f} rad ({np.degrees(angle):.1f} deg)")

# Compute positions with sample angles
positions_sample = fk.get_joint_positions(sample_joint_angles)

print("\nJoint positions with sample configuration:")
for i, pos in enumerate(positions_sample):
    label = "base" if i == 0 else fk.joint_names[i-1]
    print(f"  {label}: [{pos[0]:.4f}, {pos[1]:.4f}, {pos[2]:.4f}]")

Mapped joint angles:
  joint_1: -1.5709 rad (-90.0 deg)
  joint_2: 0.9640 rad (55.2 deg)
  joint_3: 1.9665 rad (112.7 deg)
  joint_4: 0.6598 rad (37.8 deg)
  joint_5: 1.5504 rad (88.8 deg)
  joint_6: -0.0788 rad (-4.5 deg)

Joint positions with sample configuration:
  base: [0.0000, 0.0000, 0.0000]
  joint_1: [0.0000, 0.0000, 0.1652]
  joint_2: [0.0000, 0.0000, 0.1652]
  joint_3: [-0.0001, -0.5225, 0.5280]
  joint_4: [-0.1819, -0.6341, 0.0074]
  joint_5: [-0.1819, -0.5770, -0.1111]
  joint_6: [-0.1846, -0.4555, -0.0526]


## 7. 3D Visualization

In [30]:
def plot_robot_configuration(positions: List[np.ndarray], title: str = "TM12S Configuration") -> go.Figure:
    """
    Create 3D plot of robot configuration.
    
    Args:
        positions: List of [x, y, z] joint positions
        title: Plot title
        
    Returns:
        Plotly figure
    """
    positions = np.array(positions)
    
    fig = go.Figure()
    
    # Plot links as lines
    fig.add_trace(go.Scatter3d(
        x=positions[:, 0],
        y=positions[:, 1],
        z=positions[:, 2],
        mode='lines+markers',
        line=dict(color='blue', width=8),
        marker=dict(size=8, color='red'),
        name='Robot Links'
    ))
    
    # Mark base
    fig.add_trace(go.Scatter3d(
        x=[positions[0, 0]],
        y=[positions[0, 1]],
        z=[positions[0, 2]],
        mode='markers',
        marker=dict(size=12, color='green', symbol='diamond'),
        name='Base'
    ))
    
    # Mark end-effector
    fig.add_trace(go.Scatter3d(
        x=[positions[-1, 0]],
        y=[positions[-1, 1]],
        z=[positions[-1, 2]],
        mode='markers',
        marker=dict(size=12, color='orange', symbol='diamond'),
        name='End-Effector'
    ))
    
    # Set layout
    max_range = max(
        positions[:, 0].max() - positions[:, 0].min(),
        positions[:, 1].max() - positions[:, 1].min(),
        positions[:, 2].max() - positions[:, 2].min()
    ) * 0.6
    
    center = positions.mean(axis=0)
    
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis=dict(range=[center[0]-max_range, center[0]+max_range], title='X (m)'),
            yaxis=dict(range=[center[1]-max_range, center[1]+max_range], title='Y (m)'),
            zaxis=dict(range=[center[2]-max_range, center[2]+max_range], title='Z (m)'),
            aspectmode='cube'
        ),
        width=800,
        height=600
    )
    
    return fig

In [31]:
# Plot zero configuration
fig_zero = plot_robot_configuration(positions_zero, "TM12S - Zero Configuration")
fig_zero.show()

In [33]:
# Plot sample configuration from rosbag
fig_sample = plot_robot_configuration(positions_sample, "TM12S - Sample Configuration from Rosbag")
fig_sample.show()

In [34]:
# Compare multiple configurations side by side
test_configs = [
    ("Zero", {name: 0.0 for name in fk.joint_names}),
    ("Shoulder -90deg", {**{name: 0.0 for name in fk.joint_names}, fk.joint_names[1]: -np.pi/2}),
    ("Elbow 90deg", {**{name: 0.0 for name in fk.joint_names}, fk.joint_names[2]: np.pi/2}),
    ("Sample from Bag", sample_joint_angles)
]

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{'type': 'scene'}, {'type': 'scene'}],
           [{'type': 'scene'}, {'type': 'scene'}]],
    subplot_titles=[cfg[0] for cfg in test_configs]
)

for idx, (name, joints) in enumerate(test_configs):
    row, col = divmod(idx, 2)
    positions = np.array(fk.get_joint_positions(joints))
    
    fig.add_trace(
        go.Scatter3d(
            x=positions[:, 0],
            y=positions[:, 1],
            z=positions[:, 2],
            mode='lines+markers',
            line=dict(color='blue', width=6),
            marker=dict(size=6, color='red'),
            name=name
        ),
        row=row+1, col=col+1
    )

fig.update_layout(
    title="TM12S Configuration Comparison",
    height=800,
    width=1000,
    showlegend=False
)

fig.show()

## 8. Summary and Next Steps

In [35]:
print("=" * 60)
print("PHASE 1 SETUP COMPLETE")
print("=" * 60)

print(f"\n1. URDF Extracted:")
print(f"   - Robot: {robot.name}")
print(f"   - Joints: {len(fk.joint_names)}")
print(f"   - Saved to: {urdf_output_path}")

print(f"\n2. Static Transforms Extracted:")
print(f"   - Count: {len(static_transforms)}")
print(f"   - Saved to: {transforms_output_path}")

print(f"\n3. Forward Kinematics Validated:")
print(f"   - Zero configuration tested")
print(f"   - Sample rosbag configuration tested")
print(f"   - Visualization working")

print(f"\n4. Joint Mapping (rosbag -> URDF):")
for i, name in enumerate(sample_js.name):
    urdf_name = list(sample_joint_angles.keys())[i] if i < len(sample_joint_angles) else "?"
    print(f"   - {name} -> {urdf_name}")

print(f"\n" + "=" * 60)
print("NEXT STEPS")
print("=" * 60)
print("""
1. Copy URDF to config directory on host:
   cp /home/maleen/csvdata/rapplab04/tm12s.urdf /home/maleen/git/RAPP_LAB_04/config/

2. Proceed to notebook 01_process_rosbags.ipynb to:
   - Extract and synchronize skeleton + joint data
   - Interactive skeleton selection
   - Export to CSV training format
""")

PHASE 1 SETUP COMPLETE

1. URDF Extracted:
   - Robot: tm12s
   - Joints: 6
   - Saved to: /data/processed/tm12s.urdf

2. Static Transforms Extracted:
   - Count: 8
   - Saved to: /data/processed/static_transforms.yaml

3. Forward Kinematics Validated:
   - Zero configuration tested
   - Sample rosbag configuration tested
   - Visualization working

4. Joint Mapping (rosbag -> URDF):
   - joint_1 -> joint_1
   - joint_2 -> joint_2
   - joint_3 -> joint_3
   - joint_4 -> joint_4
   - joint_5 -> joint_5
   - joint_6 -> joint_6

NEXT STEPS

1. Copy URDF to config directory on host:
   cp /home/maleen/csvdata/rapplab04/tm12s.urdf /home/maleen/git/RAPP_LAB_04/config/

2. Proceed to notebook 01_process_rosbags.ipynb to:
   - Extract and synchronize skeleton + joint data
   - Interactive skeleton selection
   - Export to CSV training format

